# 📋 Notebook 1：資料準備 + BERT 語義詐騙分類模型訓練

**專案**: AI_Voice 智慧語音詐騙檢測工具  
**目標**: 使用 ChiFraud 資料集訓練 BERT-base-chinese 進行中文詐騙文字分類  
**平台**: Kaggle GPU (T4 x2)  
**輸出**: `bert_fraud_classifier.pt` + `tokenizer/`

---

In [ ]:
# === Matplotlib 中文顯示修復 (手動路徑版) ===
!apt-get install -y fonts-wqy-microhei
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 手動強制加載字型檔，避開快取更新問題
font_path = '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
    plt.rcParams['axes.unicode_minus'] = False
    print('✅ 已手動載入字型: WenQuanYi Micro Hei')
else:
    print('❌ 找不到字型檔，請確認已執行 apt-get install')


## 1. 環境準備

In [ ]:
# === 安裝依賴 ===
!pip install -q praat-parselmouth -q opencc-python-reimplemented transformers datasets accelerate scikit-learn

import os, json, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import opencc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用裝置: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. 下載 ChiFraud 資料集

ChiFraud 是首個公開的中文網頁詐騙文字檢測資料集：
- **格式**: Tab 分隔 (Label_id \t Text)
- 59,106 筆專家標註的詐騙文字 + 352,328 筆正常文字
- 涵蓋 10 種詐騙主題 (Gambling/Whoring/Fake Credentials/...)
- 來源：[GitHub xuemingxxx/ChiFraud](https://github.com/xuemingxxx/ChiFraud)

In [ ]:
# === 下載 ChiFraud 資料集 ===
!git clone https://github.com/xuemingxxx/ChiFraud.git /tmp/ChiFraud 2>/dev/null || echo '已存在'
!ls -la /tmp/ChiFraud/dataset/

In [ ]:
# === 載入資料集 ===
# ChiFraud 使用 Tab 分隔符，欄位: Label_id (int) + Text (str)
# Label_id: 0=Normal, 1=Gambling, 2=Whoring, 3=Fake Credentials,
# 4=Fake Bank Card, 5=Prohibited Drugs, 6=Unauthorized Cash-Out,
# 7=Unauthorized Certification, 8=Fake SIM, 9=Underground Loan, 10=New

import glob

csv_files = sorted(glob.glob('/tmp/ChiFraud/dataset/*.csv'))
print(f'CSV 檔案: {csv_files}')

dfs = []
for f in csv_files:
    try:
        tmp = pd.read_csv(
            f, sep='\t',
            names=['Label_id', 'Text'],
            header=None,
            on_bad_lines='skip',
            encoding='utf-8',
            engine='python'
        )
        # 移除可能的標題行或無效行
        tmp = tmp[tmp['Label_id'].apply(lambda x: str(x).strip().isdigit())]
        tmp['Label_id'] = tmp['Label_id'].astype(int)
        print(f'  {os.path.basename(f)}: {len(tmp)} 筆')
        dfs.append(tmp)
    except Exception as e:
        print(f'  {os.path.basename(f)} 載入失敗: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'\n資料集總計: {len(df)} 筆')
print(f'欄位: {df.columns.tolist()}')
print(f'\nLabel 分布:')
print(df['Label_id'].value_counts().sort_index())
df.head(3)

## 3. 資料預處理 — 簡→繁轉換 + 清洗

In [ ]:
# === 繁簡轉換（s2twp = 簡→繁 + 台灣用語偏好）===
converter = opencc.OpenCC('s2twp')

def clean_and_convert(text):
    """清理文字並轉為繁體中文"""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'https?://\S+', '', text)          # 移除 URL
    text = re.sub(r'[【\[].{0,20}[】\]]', '', text)   # 移除微信/QQ格式
    text = re.sub(r'\s+', ' ', text).strip()          # 正規化空白
    text = converter.convert(text)                      # 簡→繁
    return text

df['text_clean'] = df['Text'].apply(clean_and_convert)
df = df[df['text_clean'].str.len() > 5].reset_index(drop=True)

print(f'清洗後: {len(df)} 筆')
print(f'\nLabel 分布:')
print(df['Label_id'].value_counts().sort_index())

In [ ]:
# === 二元分類：0=正常, 1=詐騙 ===
# ChiFraud: Label_id == 0 → 正常, 1-10 → 詐騙
FRAUD_CATEGORIES = {
    0: '正常', 1: '賭博', 2: '色情', 3: '假證件',
    4: '假銀行卡', 5: '違禁藥物', 6: '非法套現',
    7: '非法認證', 8: '假SIM卡', 9: '地下貸款', 10: '新型詐騙'
}

df['binary_label'] = (df['Label_id'] != 0).astype(int)

print(f'二元標籤分布:')
print(df['binary_label'].value_counts())
print(f'詐騙比例: {df["binary_label"].mean():.2%}')

In [ ]:
# === 平衡取樣 ===
from sklearn.utils import resample

fraud_df = df[df['binary_label'] == 1]
normal_df = df[df['binary_label'] == 0]

target_normal = min(len(normal_df), len(fraud_df) * 2)
normal_sampled = resample(normal_df, n_samples=target_normal, random_state=42)

df_balanced = pd.concat([fraud_df, normal_sampled], ignore_index=True).sample(frac=1, random_state=42)
print(f'平衡後: {len(df_balanced)} 筆')
print(df_balanced['binary_label'].value_counts())

## 4. BERT 模型訓練

In [ ]:
# === 資料切分 ===
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced['text_clean'].tolist(),
    df_balanced['binary_label'].tolist(),
    test_size=0.15, random_state=42,
    stratify=df_balanced['binary_label'].tolist()
)
print(f'訓練集: {len(train_texts)} 筆, 驗證集: {len(val_texts)} 筆')

In [ ]:
# === Tokenize ===
MODEL_NAME = 'bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding='max_length',
                          max_length=256, return_tensors='pt')
    ds = HFDataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })
    ds.set_format('torch')
    return ds

train_dataset = tokenize_data(train_texts, train_labels)
val_dataset = tokenize_data(val_texts, val_labels)
print(f'Tokenize 完成! shape: {train_dataset[0]["input_ids"].shape}')

In [ ]:
# === 初始化 + 類別加權 ===
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, hidden_dropout_prob=0.3
)

from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f'類別權重: {class_weights}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# === 訓練 ===
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary'),
        'recall': recall_score(labels, preds, average='binary'),
    }

training_args = TrainingArguments(
    output_dir='./bert_fraud_output',
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=5, weight_decay=0.01, warmup_ratio=0.1,
    load_best_model_at_end=True, metric_for_best_model='f1',
    fp16=True, dataloader_num_workers=2,
    logging_steps=100, report_to='none',
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
print(f'\n訓練完成! Loss: {train_result.training_loss:.4f}')

In [ ]:
# === 評估 ===
eval_result = trainer.evaluate()
print('\n=== 驗證集評估 ===')
for k, v in eval_result.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# === 混淆矩陣 ===
preds = trainer.predict(val_dataset)
y_pred = preds.predictions.argmax(-1)
y_true = val_labels

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['正常', '詐騙'], yticklabels=['正常', '詐騙'])
plt.xlabel('預測'); plt.ylabel('實際')
plt.title('BERT 中文詐騙分類 — 混淆矩陣')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

print('\n=== 分類報告 ===')
print(classification_report(y_true, y_pred, target_names=['正常', '詐騙']))

## 5. 匯出模型

In [ ]:
# === 儲存 ===
OUTPUT_DIR = './bert_fraud_classifier'
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f'{OUTPUT_DIR}/training_log.json', 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'dataset': 'ChiFraud',
        'train_size': len(train_texts), 'val_size': len(val_texts),
        'best_f1': eval_result.get('eval_f1', 0),
        'best_accuracy': eval_result.get('eval_accuracy', 0),
    }, f, indent=2)

!tar -czf bert_fraud_classifier.tar.gz -C {OUTPUT_DIR} .
print(f'模型已儲存並打包! 放置路徑: AI_Voice/models/semantic/bert_fraud_classifier/')

## 6. 推理測試

In [ ]:
model.eval()
test_sentences = [
    '這裡是公安局，你的銀行帳戶涉嫌洗錢，需要配合調查',
    '恭喜你中了大獎，請先支付手續費到指定帳戶',
    '你好，我是你的快遞員，你的包裹今天會送到',
    '明天天氣不錯，我們去公園散步吧',
    '你的訂單出現異常，需要你提供驗證碼來確認退款',
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=256, padding='max_length').to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)
        pred = probs[0][1].item()
    icon = '🔴' if pred > 0.7 else '🟡' if pred > 0.4 else '🟢'
    print(f'{icon} [{pred:.1%}] {text[:40]}...')